In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 32 — Alerting + Notification Routing
# Cell 1 — Alert table setup + config-driven recipient routing
# ─────────────────────────────────────────────────────────────

import json
import uuid
import platform
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    DoubleType,
    TimestampType
)

# ─────────────────────────────────────────────────────────────
# Product config
# ─────────────────────────────────────────────────────────────

PROJECT_NAME = "SupplySage AI"
NOTEBOOK_NAME = "32_alerting_notification_routing"
ALERTING_VERSION = "v1_config_driven_alert_routing"
CREATED_BY = "Vigya"

GOLD_SCHEMA = "supplysage_gold"

ALERT_RUN_ID = f"alerting_setup_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Core alerting tables
ALERT_ROUTING_CONFIG_TABLE = f"{GOLD_SCHEMA}.gold_alert_routing_config"
ALERT_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_alert_events"
ALERT_INBOX_TABLE = f"{GOLD_SCHEMA}.gold_alert_inbox"
ALERT_DELIVERY_LOG_TABLE = f"{GOLD_SCHEMA}.gold_alert_delivery_log"
ALERT_RUN_SUMMARY_TABLE = f"{GOLD_SCHEMA}.gold_alerting_run_summary"

# Upstream tables this notebook will use in later cells
UPSTREAM_TABLES = {
    "supplier_risk_scores": f"{GOLD_SCHEMA}.gold_supplier_risk_scores",
    "sku_stockout_risk_scores": f"{GOLD_SCHEMA}.gold_sku_stockout_risk_scores",
    "external_risk_event_mart": f"{GOLD_SCHEMA}.gold_external_risk_event_mart",
    "chat_context_snapshots": f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
    "workflow_deployment_summary": f"{GOLD_SCHEMA}.gold_workflow_deployment_summary",
    "workflow_run_status_snapshots": f"{GOLD_SCHEMA}.gold_workflow_job_run_status_snapshots",
    "serving_health": f"{GOLD_SCHEMA}.gold_supplier_risk_agent_serving_health",
    "serving_summary": f"{GOLD_SCHEMA}.gold_supplier_risk_agent_serving_summary"
}

print("ALERT_RUN_ID:", ALERT_RUN_ID)
print("ALERTING_VERSION:", ALERTING_VERSION)
print("GOLD_SCHEMA:", GOLD_SCHEMA)

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def safe_row_count(table_name: str):
    try:
        return spark.table(table_name).count()
    except Exception:
        return None


def create_empty_delta_table(table_name: str, schema: StructType):
    """
    Creates an empty Delta table if it does not already exist.
    Does not overwrite existing alert data.
    """
    if not table_exists(table_name):
        empty_df = spark.createDataFrame([], schema=schema)

        (
            empty_df
            .write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .saveAsTable(table_name)
        )

        print(f"Created table: {table_name}")
    else:
        print(f"Table already exists: {table_name}")


# ─────────────────────────────────────────────────────────────
# Alert routing config
# ─────────────────────────────────────────────────────────────
# Important product design:
# Recipients are NOT hardcoded inside alert generation logic.
# They are seeded into a Delta config table and later joined based on
# alert_type, severity, risk_category, supplier_id, sku_id, and active flag.

routing_config_schema = StructType([
    StructField("routing_rule_id", StringType(), False),
    StructField("routing_config_version", StringType(), True),
    StructField("rule_priority", IntegerType(), True),

    StructField("alert_type", StringType(), True),
    StructField("risk_category", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("minimum_severity_rank", IntegerType(), True),

    StructField("supplier_id", StringType(), True),
    StructField("sku_id", StringType(), True),

    StructField("recipient_group", StringType(), True),
    StructField("recipient_name", StringType(), True),
    StructField("recipient_email", StringType(), True),
    StructField("channel", StringType(), True),

    StructField("send_enabled", BooleanType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("escalation_minutes", IntegerType(), True),

    StructField("notes", StringType(), True),
    StructField("created_by", StringType(), True),
    StructField("created_at", TimestampType(), True),
])

config_created_at = datetime.now(timezone.utc)

routing_config_rows = [
    # Compliance / sanctions alerts
    {
        "routing_rule_id": "route_sanctions_high_compliance_v1",
        "routing_config_version": ALERTING_VERSION,
        "rule_priority": 10,
        "alert_type": "external_event",
        "risk_category": "sanctions_compliance",
        "severity": "high",
        "minimum_severity_rank": 3,
        "supplier_id": "*",
        "sku_id": "*",
        "recipient_group": "compliance_team",
        "recipient_name": "Vigya Awasthi",
        "recipient_email": "vigya.awasthi@tamu.edu",
        "channel": "email",
        "send_enabled": False,
        "is_active": True,
        "escalation_minutes": 60,
        "notes": "Dev/test recipient for sanctions and compliance alerts.",
        "created_by": CREATED_BY,
        "created_at": config_created_at
    },

    # Cyber / CISA alerts
    {
        "routing_rule_id": "route_cyber_high_security_v1",
        "routing_config_version": ALERTING_VERSION,
        "rule_priority": 20,
        "alert_type": "external_event",
        "risk_category": "cyber",
        "severity": "high",
        "minimum_severity_rank": 3,
        "supplier_id": "*",
        "sku_id": "*",
        "recipient_group": "security_team",
        "recipient_name": "Siddharath Prakash Kumar Bohra",
        "recipient_email": "siddharath.bohra@tamu.edu",
        "channel": "email",
        "send_enabled": False,
        "is_active": True,
        "escalation_minutes": 120,
        "notes": "Dev/test recipient for cyber and CISA-related supplier alerts.",
        "created_by": CREATED_BY,
        "created_at": config_created_at
    },

    # Supplier risk alerts
    {
        "routing_rule_id": "route_supplier_high_ops_v1",
        "routing_config_version": ALERTING_VERSION,
        "rule_priority": 30,
        "alert_type": "supplier_risk",
        "risk_category": "supplier_risk",
        "severity": "high",
        "minimum_severity_rank": 3,
        "supplier_id": "*",
        "sku_id": "*",
        "recipient_group": "supplier_ops_team",
        "recipient_name": "Vigya Awasthi",
        "recipient_email": "vigya.awasthi@tamu.edu",
        "channel": "email",
        "send_enabled": False,
        "is_active": True,
        "escalation_minutes": 180,
        "notes": "Dev/test recipient for supplier high-risk threshold alerts.",
        "created_by": CREATED_BY,
        "created_at": config_created_at
    },

    # SKU stockout alerts
    {
        "routing_rule_id": "route_sku_stockout_high_inventory_v1",
        "routing_config_version": ALERTING_VERSION,
        "rule_priority": 40,
        "alert_type": "sku_stockout",
        "risk_category": "inventory_stockout",
        "severity": "high",
        "minimum_severity_rank": 3,
        "supplier_id": "*",
        "sku_id": "*",
        "recipient_group": "inventory_team",
        "recipient_name": "Siddharath Prakash Kumar Bohra",
        "recipient_email": "siddharath.bohra@tamu.edu",
        "channel": "email",
        "send_enabled": False,
        "is_active": True,
        "escalation_minutes": 180,
        "notes": "Dev/test recipient for high SKU stockout risk alerts.",
        "created_by": CREATED_BY,
        "created_at": config_created_at
    },

    # Weather / logistics / operational alerts
    {
        "routing_rule_id": "route_logistics_medium_ops_v1",
        "routing_config_version": ALERTING_VERSION,
        "rule_priority": 50,
        "alert_type": "external_event",
        "risk_category": "logistics",
        "severity": "medium",
        "minimum_severity_rank": 2,
        "supplier_id": "*",
        "sku_id": "*",
        "recipient_group": "supply_chain_ops_team",
        "recipient_name": "Vigya Awasthi",
        "recipient_email": "vigya.awasthi@tamu.edu",
        "channel": "email",
        "send_enabled": False,
        "is_active": True,
        "escalation_minutes": 240,
        "notes": "Dev/test recipient for logistics, weather, and operational disruption alerts.",
        "created_by": CREATED_BY,
        "created_at": config_created_at
    },

    # Lakeflow / data platform alerts
    {
        "routing_rule_id": "route_lakeflow_failure_data_platform_v1",
        "routing_config_version": ALERTING_VERSION,
        "rule_priority": 60,
        "alert_type": "workflow_health",
        "risk_category": "data_platform",
        "severity": "critical",
        "minimum_severity_rank": 4,
        "supplier_id": "*",
        "sku_id": "*",
        "recipient_group": "data_platform_team",
        "recipient_name": "Vigya Awasthi",
        "recipient_email": "vigya.awasthi@tamu.edu",
        "channel": "email",
        "send_enabled": False,
        "is_active": True,
        "escalation_minutes": 30,
        "notes": "Dev/test recipient for Lakeflow job failure alerts.",
        "created_by": CREATED_BY,
        "created_at": config_created_at
    },

    # Agent / serving health alerts
    {
        "routing_rule_id": "route_agent_serving_failure_ai_platform_v1",
        "routing_config_version": ALERTING_VERSION,
        "rule_priority": 70,
        "alert_type": "agent_serving_health",
        "risk_category": "ai_platform",
        "severity": "critical",
        "minimum_severity_rank": 4,
        "supplier_id": "*",
        "sku_id": "*",
        "recipient_group": "ai_platform_team",
        "recipient_name": "Siddharath Prakash Kumar Bohra",
        "recipient_email": "siddharath.bohra@tamu.edu",
        "channel": "email",
        "send_enabled": False,
        "is_active": True,
        "escalation_minutes": 30,
        "notes": "Dev/test recipient for agent serving and evaluation health alerts.",
        "created_by": CREATED_BY,
        "created_at": config_created_at
    },

    # Default fallback route
    {
        "routing_rule_id": "route_default_high_v1",
        "routing_config_version": ALERTING_VERSION,
        "rule_priority": 999,
        "alert_type": "*",
        "risk_category": "*",
        "severity": "high",
        "minimum_severity_rank": 3,
        "supplier_id": "*",
        "sku_id": "*",
        "recipient_group": "default_alert_team",
        "recipient_name": "Vigya Awasthi",
        "recipient_email": "vigya.awasthi@tamu.edu",
        "channel": "email",
        "send_enabled": False,
        "is_active": True,
        "escalation_minutes": 240,
        "notes": "Default fallback recipient for high-severity alerts with no more specific route.",
        "created_by": CREATED_BY,
        "created_at": config_created_at
    }
]

routing_config_df = spark.createDataFrame(
    routing_config_rows,
    schema=routing_config_schema
)

# Create routing table if missing, then merge config rows by routing_rule_id.
create_empty_delta_table(ALERT_ROUTING_CONFIG_TABLE, routing_config_schema)

routing_config_df.createOrReplaceTempView("seed_alert_routing_config")

spark.sql(f"""
MERGE INTO {ALERT_ROUTING_CONFIG_TABLE} AS target
USING seed_alert_routing_config AS source
ON target.routing_rule_id = source.routing_rule_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print(f"Seeded routing config into: {ALERT_ROUTING_CONFIG_TABLE}")

# ─────────────────────────────────────────────────────────────
# Alert events table schema
# ─────────────────────────────────────────────────────────────

alert_events_schema = StructType([
    StructField("alert_id", StringType(), False),
    StructField("alert_run_id", StringType(), True),
    StructField("alert_type", StringType(), True),
    StructField("alert_source", StringType(), True),
    StructField("risk_category", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("severity_rank", IntegerType(), True),

    StructField("supplier_id", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("sku_id", StringType(), True),
    StructField("product_name", StringType(), True),

    StructField("risk_score", DoubleType(), True),
    StructField("risk_band", StringType(), True),
    StructField("trigger_metric_name", StringType(), True),
    StructField("trigger_metric_value", DoubleType(), True),
    StructField("trigger_threshold", DoubleType(), True),

    StructField("event_date", StringType(), True),
    StructField("event_source", StringType(), True),
    StructField("event_title", StringType(), True),
    StructField("event_url", StringType(), True),

    StructField("alert_title", StringType(), True),
    StructField("alert_message", StringType(), True),
    StructField("recommended_action", StringType(), True),

    StructField("dedupe_key", StringType(), True),
    StructField("alert_status", StringType(), True),
    StructField("is_active", BooleanType(), True),

    StructField("created_at", TimestampType(), True),
    StructField("created_by", StringType(), True),
    StructField("source_payload_json", StringType(), True),
])

create_empty_delta_table(ALERT_EVENTS_TABLE, alert_events_schema)

# ─────────────────────────────────────────────────────────────
# Alert inbox table schema
# ─────────────────────────────────────────────────────────────

alert_inbox_schema = StructType([
    StructField("inbox_item_id", StringType(), False),
    StructField("alert_id", StringType(), True),
    StructField("alert_run_id", StringType(), True),
    StructField("alert_type", StringType(), True),
    StructField("risk_category", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("severity_rank", IntegerType(), True),

    StructField("supplier_id", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("sku_id", StringType(), True),

    StructField("recipient_group", StringType(), True),
    StructField("recipient_name", StringType(), True),
    StructField("recipient_email", StringType(), True),
    StructField("channel", StringType(), True),

    StructField("alert_title", StringType(), True),
    StructField("alert_message", StringType(), True),
    StructField("recommended_action", StringType(), True),

    StructField("inbox_status", StringType(), True),
    StructField("assigned_to", StringType(), True),
    StructField("acknowledged_at", TimestampType(), True),
    StructField("resolved_at", TimestampType(), True),

    StructField("created_at", TimestampType(), True),
    StructField("created_by", StringType(), True),
])

create_empty_delta_table(ALERT_INBOX_TABLE, alert_inbox_schema)

# ─────────────────────────────────────────────────────────────
# Delivery log table schema
# ─────────────────────────────────────────────────────────────

delivery_log_schema = StructType([
    StructField("delivery_id", StringType(), False),
    StructField("alert_id", StringType(), True),
    StructField("inbox_item_id", StringType(), True),
    StructField("alert_run_id", StringType(), True),

    StructField("recipient_group", StringType(), True),
    StructField("recipient_name", StringType(), True),
    StructField("recipient_email", StringType(), True),
    StructField("channel", StringType(), True),

    StructField("send_enabled", BooleanType(), True),
    StructField("delivery_status", StringType(), True),
    StructField("delivery_attempted", BooleanType(), True),
    StructField("delivery_error", StringType(), True),

    StructField("message_subject", StringType(), True),
    StructField("message_body", StringType(), True),

    StructField("created_at", TimestampType(), True),
    StructField("delivered_at", TimestampType(), True),
    StructField("created_by", StringType(), True),
])

create_empty_delta_table(ALERT_DELIVERY_LOG_TABLE, delivery_log_schema)

# ─────────────────────────────────────────────────────────────
# Run summary table schema
# ─────────────────────────────────────────────────────────────

run_summary_schema = StructType([
    StructField("alert_run_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("alerting_version", StringType(), True),
    StructField("routing_config_table", StringType(), True),
    StructField("alert_events_table", StringType(), True),
    StructField("alert_inbox_table", StringType(), True),
    StructField("alert_delivery_log_table", StringType(), True),

    StructField("routing_rules_active", IntegerType(), True),
    StructField("upstream_tables_checked", IntegerType(), True),
    StructField("upstream_tables_available", IntegerType(), True),

    StructField("setup_status", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("created_by", StringType(), True),
    StructField("environment_json", StringType(), True),
])

create_empty_delta_table(ALERT_RUN_SUMMARY_TABLE, run_summary_schema)

# ─────────────────────────────────────────────────────────────
# Upstream table availability check
# ─────────────────────────────────────────────────────────────

upstream_check_rows = []

for key, table_name in UPSTREAM_TABLES.items():
    exists_flag = table_exists(table_name)
    row_count = safe_row_count(table_name) if exists_flag else None

    upstream_check_rows.append({
        "upstream_key": key,
        "table_name": table_name,
        "exists": bool(exists_flag),
        "row_count": int(row_count) if row_count is not None else None
    })

print("\nUpstream alert source table checks:")
display(spark.createDataFrame(upstream_check_rows))

upstream_tables_checked = len(upstream_check_rows)
upstream_tables_available = sum(1 for row in upstream_check_rows if row["exists"])

routing_rules_active = (
    spark.table(ALERT_ROUTING_CONFIG_TABLE)
    .filter(F.col("is_active") == True)
    .count()
)

setup_status = (
    "alerting_tables_ready"
    if routing_rules_active > 0 and upstream_tables_available > 0
    else "alerting_setup_incomplete"
)

summary_rows = [{
    "alert_run_id": ALERT_RUN_ID,
    "project_name": PROJECT_NAME,
    "alerting_version": ALERTING_VERSION,
    "routing_config_table": ALERT_ROUTING_CONFIG_TABLE,
    "alert_events_table": ALERT_EVENTS_TABLE,
    "alert_inbox_table": ALERT_INBOX_TABLE,
    "alert_delivery_log_table": ALERT_DELIVERY_LOG_TABLE,
    "routing_rules_active": int(routing_rules_active),
    "upstream_tables_checked": int(upstream_tables_checked),
    "upstream_tables_available": int(upstream_tables_available),
    "setup_status": setup_status,
    "created_at": datetime.now(timezone.utc),
    "created_by": CREATED_BY,
    "environment_json": json.dumps({
        "python_version": platform.python_version(),
        "notebook_name": NOTEBOOK_NAME,
        "alert_run_id": ALERT_RUN_ID
    }, indent=2)
}]

summary_df = spark.createDataFrame(summary_rows, schema=run_summary_schema)

(
    summary_df
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(ALERT_RUN_SUMMARY_TABLE)
)

# ─────────────────────────────────────────────────────────────
# Preview routing config
# ─────────────────────────────────────────────────────────────

print("\nAlerting setup status:", setup_status)
print("Active routing rules:", routing_rules_active)

print("\nRouting config preview:")
display(
    spark.table(ALERT_ROUTING_CONFIG_TABLE)
    .filter(F.col("is_active") == True)
    .select(
        "rule_priority",
        "alert_type",
        "risk_category",
        "severity",
        "minimum_severity_rank",
        "recipient_group",
        "recipient_name",
        "recipient_email",
        "channel",
        "send_enabled",
        "is_active"
    )
    .orderBy("rule_priority")
)

print("\nCell 1 complete.")
print("Next: Cell 2 will generate alert events from supplier risk, SKU stockout risk, external events, Lakeflow health, and serving health.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 32 — Cell 2
# Generate alert events from supplier, SKU, external, workflow, and serving signals
# ─────────────────────────────────────────────────────────────

import json
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    DoubleType,
    TimestampType
)

assert "ALERT_RUN_ID" in globals(), "ALERT_RUN_ID missing. Rerun Cell 1."
assert "ALERT_EVENTS_TABLE" in globals(), "ALERT_EVENTS_TABLE missing. Rerun Cell 1."
assert "UPSTREAM_TABLES" in globals(), "UPSTREAM_TABLES missing. Rerun Cell 1."

alert_generation_started_at = datetime.now(timezone.utc)

SEVERITY_RANK = {
    "low": 1,
    "medium": 2,
    "high": 3,
    "critical": 4
}

SUPPLIER_HIGH_RISK_THRESHOLD = 70.0
SUPPLIER_MEDIUM_RISK_THRESHOLD = 45.0

SKU_HIGH_STOCKOUT_THRESHOLD = 0.75
SKU_MEDIUM_STOCKOUT_THRESHOLD = 0.50

MAX_ALERTS_PER_TYPE = 50

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def get_first_existing_col(columns, candidates):
    for col_name in candidates:
        if col_name in columns:
            return col_name
    return None


def normalize_severity_expr(col_name):
    return (
        F.when(F.lower(F.col(col_name)).isin("critical"), F.lit("critical"))
        .when(F.lower(F.col(col_name)).isin("high"), F.lit("high"))
        .when(F.lower(F.col(col_name)).isin("medium"), F.lit("medium"))
        .when(F.lower(F.col(col_name)).isin("low"), F.lit("low"))
        .otherwise(F.lit("medium"))
    )


def severity_rank_expr(severity_col):
    return (
        F.when(F.col(severity_col) == "critical", F.lit(4))
        .when(F.col(severity_col) == "high", F.lit(3))
        .when(F.col(severity_col) == "medium", F.lit(2))
        .when(F.col(severity_col) == "low", F.lit(1))
        .otherwise(F.lit(2))
    )


def add_common_alert_columns(df):
    return (
        df
        .withColumn("alert_run_id", F.lit(ALERT_RUN_ID))
        .withColumn("created_at", F.current_timestamp())
        .withColumn("created_by", F.lit(CREATED_BY))
        .withColumn("alert_status", F.lit("new"))
        .withColumn("is_active", F.lit(True))
    )


def empty_alert_df():
    schema = spark.table(ALERT_EVENTS_TABLE).schema
    return spark.createDataFrame([], schema=schema)


def align_to_alert_schema(df):
    """
    Align a generated alert DataFrame to the existing alert event table schema.
    Adds any missing columns as null.
    """
    target_schema = spark.table(ALERT_EVENTS_TABLE).schema
    target_cols = [field.name for field in target_schema]

    aligned_df = df

    for field in target_schema:
        if field.name not in aligned_df.columns:
            aligned_df = aligned_df.withColumn(field.name, F.lit(None).cast(field.dataType))
        else:
            aligned_df = aligned_df.withColumn(field.name, F.col(field.name).cast(field.dataType))

    return aligned_df.select(target_cols)


def stable_alert_id_expr(dedupe_col):
    return F.concat(F.lit("alert_"), F.sha2(F.col(dedupe_col), 256).substr(1, 24))


# ─────────────────────────────────────────────────────────────
# 1. Supplier risk alerts
# ─────────────────────────────────────────────────────────────

supplier_alerts_df = empty_alert_df()

supplier_table = UPSTREAM_TABLES["supplier_risk_scores"]

if table_exists(supplier_table):
    supplier_df = spark.table(supplier_table)
    supplier_cols = supplier_df.columns

    supplier_id_col = get_first_existing_col(supplier_cols, ["supplier_id"])
    supplier_name_col = get_first_existing_col(supplier_cols, ["supplier_name"])
    risk_score_col = get_first_existing_col(supplier_cols, ["supplier_risk_score", "risk_score", "final_supplier_risk_score"])
    risk_band_col = get_first_existing_col(supplier_cols, ["risk_band", "supplier_risk_band"])
    driver_col = get_first_existing_col(supplier_cols, ["top_risk_driver", "final_top_risk_driver"])
    action_col = get_first_existing_col(supplier_cols, ["recommended_action", "final_recommended_action"])

    if supplier_id_col and risk_score_col:
        base = supplier_df

        if supplier_name_col is None:
            base = base.withColumn("supplier_name_tmp", F.col(supplier_id_col))
            supplier_name_col = "supplier_name_tmp"

        if risk_band_col is None:
            base = base.withColumn(
                "risk_band_tmp",
                F.when(F.col(risk_score_col) >= SUPPLIER_HIGH_RISK_THRESHOLD, F.lit("high"))
                .when(F.col(risk_score_col) >= SUPPLIER_MEDIUM_RISK_THRESHOLD, F.lit("medium"))
                .otherwise(F.lit("low"))
            )
            risk_band_col = "risk_band_tmp"

        if driver_col is None:
            base = base.withColumn("top_risk_driver_tmp", F.lit("Supplier risk threshold"))
            driver_col = "top_risk_driver_tmp"

        if action_col is None:
            base = base.withColumn("recommended_action_tmp", F.lit("Review supplier risk drivers and mitigation options."))
            action_col = "recommended_action_tmp"

        supplier_alerts_df = (
            base
            .filter(F.col(risk_score_col) >= F.lit(SUPPLIER_HIGH_RISK_THRESHOLD))
            .orderBy(F.col(risk_score_col).desc())
            .limit(MAX_ALERTS_PER_TYPE)
            .withColumn("alert_type", F.lit("supplier_risk"))
            .withColumn("alert_source", F.lit(supplier_table))
            .withColumn("risk_category", F.lit("supplier_risk"))
            .withColumn("severity", F.lit("high"))
            .withColumn("severity_rank", F.lit(3))
            .withColumn("supplier_id", F.col(supplier_id_col).cast("string"))
            .withColumn("supplier_name", F.col(supplier_name_col).cast("string"))
            .withColumn("sku_id", F.lit(None).cast("string"))
            .withColumn("product_name", F.lit(None).cast("string"))
            .withColumn("risk_score", F.col(risk_score_col).cast("double"))
            .withColumn("risk_band", F.col(risk_band_col).cast("string"))
            .withColumn("trigger_metric_name", F.lit("supplier_risk_score"))
            .withColumn("trigger_metric_value", F.col(risk_score_col).cast("double"))
            .withColumn("trigger_threshold", F.lit(float(SUPPLIER_HIGH_RISK_THRESHOLD)))
            .withColumn("event_date", F.current_date().cast("string"))
            .withColumn("event_source", F.lit("supplier_risk_score_model"))
            .withColumn("event_title", F.concat(F.lit("High supplier risk: "), F.col(supplier_name_col).cast("string")))
            .withColumn("event_url", F.lit(None).cast("string"))
            .withColumn(
                "alert_title",
                F.concat(F.lit("High supplier risk detected for "), F.col(supplier_name_col).cast("string"))
            )
            .withColumn(
                "alert_message",
                F.concat(
                    F.lit("Supplier "),
                    F.col(supplier_name_col).cast("string"),
                    F.lit(" has risk score "),
                    F.round(F.col(risk_score_col), 2).cast("string"),
                    F.lit(". Top driver: "),
                    F.col(driver_col).cast("string")
                )
            )
            .withColumn("recommended_action", F.col(action_col).cast("string"))
            .withColumn(
                "dedupe_key",
                F.concat_ws("|", F.lit("supplier_risk"), F.col(supplier_id_col).cast("string"), F.current_date().cast("string"))
            )
            .withColumn("alert_id", stable_alert_id_expr("dedupe_key"))
        )

        supplier_alerts_df = add_common_alert_columns(supplier_alerts_df)
        supplier_alerts_df = align_to_alert_schema(supplier_alerts_df)

print("Supplier alerts generated:", supplier_alerts_df.count())

# ─────────────────────────────────────────────────────────────
# 2. SKU stockout alerts
# ─────────────────────────────────────────────────────────────

sku_alerts_df = empty_alert_df()

sku_table = UPSTREAM_TABLES["sku_stockout_risk_scores"]

if table_exists(sku_table):
    sku_df = spark.table(sku_table)
    sku_cols = sku_df.columns

    sku_id_col = get_first_existing_col(sku_cols, ["sku_id", "canonical_sku_id", "item_id"])
    product_name_col = get_first_existing_col(sku_cols, ["product_name", "item_name", "sku_name"])
    supplier_id_col = get_first_existing_col(sku_cols, ["supplier_id", "primary_supplier_id"])
    supplier_name_col = get_first_existing_col(sku_cols, ["supplier_name", "primary_supplier_name"])
    stockout_score_col = get_first_existing_col(sku_cols, ["sku_stockout_risk_score", "stockout_risk_score", "risk_score"])
    risk_band_col = get_first_existing_col(sku_cols, ["stockout_risk_band", "risk_band"])

    if sku_id_col and stockout_score_col:
        base = sku_df

        if product_name_col is None:
            base = base.withColumn("product_name_tmp", F.col(sku_id_col))
            product_name_col = "product_name_tmp"

        if supplier_id_col is None:
            base = base.withColumn("supplier_id_tmp", F.lit(None).cast("string"))
            supplier_id_col = "supplier_id_tmp"

        if supplier_name_col is None:
            base = base.withColumn("supplier_name_tmp", F.lit(None).cast("string"))
            supplier_name_col = "supplier_name_tmp"

        if risk_band_col is None:
            base = base.withColumn(
                "risk_band_tmp",
                F.when(F.col(stockout_score_col) >= SKU_HIGH_STOCKOUT_THRESHOLD, F.lit("high"))
                .when(F.col(stockout_score_col) >= SKU_MEDIUM_STOCKOUT_THRESHOLD, F.lit("medium"))
                .otherwise(F.lit("low"))
            )
            risk_band_col = "risk_band_tmp"

        sku_alerts_df = (
            base
            .filter(F.col(stockout_score_col) >= F.lit(SKU_HIGH_STOCKOUT_THRESHOLD))
            .orderBy(F.col(stockout_score_col).desc())
            .limit(MAX_ALERTS_PER_TYPE)
            .withColumn("alert_type", F.lit("sku_stockout"))
            .withColumn("alert_source", F.lit(sku_table))
            .withColumn("risk_category", F.lit("inventory_stockout"))
            .withColumn("severity", F.lit("high"))
            .withColumn("severity_rank", F.lit(3))
            .withColumn("supplier_id", F.col(supplier_id_col).cast("string"))
            .withColumn("supplier_name", F.col(supplier_name_col).cast("string"))
            .withColumn("sku_id", F.col(sku_id_col).cast("string"))
            .withColumn("product_name", F.col(product_name_col).cast("string"))
            .withColumn("risk_score", F.col(stockout_score_col).cast("double"))
            .withColumn("risk_band", F.col(risk_band_col).cast("string"))
            .withColumn("trigger_metric_name", F.lit("sku_stockout_risk_score"))
            .withColumn("trigger_metric_value", F.col(stockout_score_col).cast("double"))
            .withColumn("trigger_threshold", F.lit(float(SKU_HIGH_STOCKOUT_THRESHOLD)))
            .withColumn("event_date", F.current_date().cast("string"))
            .withColumn("event_source", F.lit("sku_stockout_risk_model"))
            .withColumn("event_title", F.concat(F.lit("High SKU stockout risk: "), F.col(sku_id_col).cast("string")))
            .withColumn("event_url", F.lit(None).cast("string"))
            .withColumn(
                "alert_title",
                F.concat(F.lit("High stockout risk detected for SKU "), F.col(sku_id_col).cast("string"))
            )
            .withColumn(
                "alert_message",
                F.concat(
                    F.lit("SKU "),
                    F.col(sku_id_col).cast("string"),
                    F.lit(" has stockout risk score "),
                    F.round(F.col(stockout_score_col), 3).cast("string"),
                    F.lit(".")
                )
            )
            .withColumn("recommended_action", F.lit("Review inventory position, supplier dependency, and alternate sourcing options."))
            .withColumn(
                "dedupe_key",
                F.concat_ws("|", F.lit("sku_stockout"), F.col(sku_id_col).cast("string"), F.current_date().cast("string"))
            )
            .withColumn("alert_id", stable_alert_id_expr("dedupe_key"))
        )

        sku_alerts_df = add_common_alert_columns(sku_alerts_df)
        sku_alerts_df = align_to_alert_schema(sku_alerts_df)

print("SKU stockout alerts generated:", sku_alerts_df.count())

# ─────────────────────────────────────────────────────────────
# 3. External event alerts
# ─────────────────────────────────────────────────────────────

external_alerts_df = empty_alert_df()

external_table = UPSTREAM_TABLES["external_risk_event_mart"]

if table_exists(external_table):
    ext_df = spark.table(external_table)
    ext_cols = ext_df.columns

    supplier_id_col = get_first_existing_col(ext_cols, ["supplier_id"])
    supplier_name_col = get_first_existing_col(ext_cols, ["supplier_name"])
    risk_category_col = get_first_existing_col(ext_cols, ["risk_category", "event_category", "category"])
    severity_col = get_first_existing_col(ext_cols, ["severity", "event_severity"])
    event_date_col = get_first_existing_col(ext_cols, ["event_date", "published_at", "created_at"])
    event_source_col = get_first_existing_col(ext_cols, ["source_name", "event_source", "source_system", "driver_source"])
    event_title_col = get_first_existing_col(ext_cols, ["event_title", "title", "headline"])
    event_url_col = get_first_existing_col(ext_cols, ["event_url", "url", "source_url"])
    event_id_col = get_first_existing_col(ext_cols, ["event_id", "external_event_id", "evidence_id"])

    if supplier_id_col and risk_category_col and severity_col:
        base = ext_df

        if supplier_name_col is None:
            base = base.withColumn("supplier_name_tmp", F.col(supplier_id_col))
            supplier_name_col = "supplier_name_tmp"

        if event_date_col is None:
            base = base.withColumn("event_date_tmp", F.current_date().cast("string"))
            event_date_col = "event_date_tmp"

        if event_source_col is None:
            base = base.withColumn("event_source_tmp", F.lit("external_risk_event_mart"))
            event_source_col = "event_source_tmp"

        if event_title_col is None:
            base = base.withColumn("event_title_tmp", F.concat(F.lit("External risk event: "), F.col(risk_category_col).cast("string")))
            event_title_col = "event_title_tmp"

        if event_url_col is None:
            base = base.withColumn("event_url_tmp", F.lit(None).cast("string"))
            event_url_col = "event_url_tmp"

        if event_id_col is None:
            base = base.withColumn(
                "event_id_tmp",
                F.sha2(
                    F.concat_ws(
                        "|",
                        F.col(supplier_id_col).cast("string"),
                        F.col(risk_category_col).cast("string"),
                        F.col(event_title_col).cast("string"),
                        F.col(event_date_col).cast("string")
                    ),
                    256
                )
            )
            event_id_col = "event_id_tmp"

        base = (
            base
            .withColumn("normalized_severity", normalize_severity_expr(severity_col))
            .withColumn("severity_rank_tmp", severity_rank_expr("normalized_severity"))
        )

        external_alerts_df = (
            base
            .filter(F.col("severity_rank_tmp") >= F.lit(3))
            .orderBy(F.col("severity_rank_tmp").desc(), F.col(event_date_col).desc())
            .limit(MAX_ALERTS_PER_TYPE)
            .withColumn("alert_type", F.lit("external_event"))
            .withColumn("alert_source", F.lit(external_table))
            .withColumn("risk_category", F.col(risk_category_col).cast("string"))
            .withColumn("severity", F.col("normalized_severity"))
            .withColumn("severity_rank", F.col("severity_rank_tmp").cast("int"))
            .withColumn("supplier_id", F.col(supplier_id_col).cast("string"))
            .withColumn("supplier_name", F.col(supplier_name_col).cast("string"))
            .withColumn("sku_id", F.lit(None).cast("string"))
            .withColumn("product_name", F.lit(None).cast("string"))
            .withColumn("risk_score", F.lit(None).cast("double"))
            .withColumn("risk_band", F.col("normalized_severity"))
            .withColumn("trigger_metric_name", F.lit("external_event_severity_rank"))
            .withColumn("trigger_metric_value", F.col("severity_rank_tmp").cast("double"))
            .withColumn("trigger_threshold", F.lit(3.0))
            .withColumn("event_date", F.col(event_date_col).cast("string"))
            .withColumn("event_source", F.col(event_source_col).cast("string"))
            .withColumn("event_title", F.col(event_title_col).cast("string"))
            .withColumn("event_url", F.col(event_url_col).cast("string"))
            .withColumn(
                "alert_title",
                F.concat(
                    F.lit("High-severity external event for "),
                    F.col(supplier_name_col).cast("string")
                )
            )
            .withColumn(
                "alert_message",
                F.concat(
                    F.lit("External "),
                    F.col(risk_category_col).cast("string"),
                    F.lit(" event detected with severity "),
                    F.col("normalized_severity"),
                    F.lit(": "),
                    F.col(event_title_col).cast("string")
                )
            )
            .withColumn("recommended_action", F.lit("Review external event evidence and assess supplier/SKU exposure."))
            .withColumn(
                "dedupe_key",
                F.concat_ws(
                    "|",
                    F.lit("external_event"),
                    F.col(supplier_id_col).cast("string"),
                    F.col(event_id_col).cast("string")
                )
            )
            .withColumn("alert_id", stable_alert_id_expr("dedupe_key"))
        )

        external_alerts_df = add_common_alert_columns(external_alerts_df)
        external_alerts_df = align_to_alert_schema(external_alerts_df)

print("External event alerts generated:", external_alerts_df.count())

# ─────────────────────────────────────────────────────────────
# 4. Workflow health alerts
# ─────────────────────────────────────────────────────────────

workflow_alerts_df = empty_alert_df()

workflow_status_table = UPSTREAM_TABLES["workflow_run_status_snapshots"]

if table_exists(workflow_status_table):
    workflow_df = spark.table(workflow_status_table)
    workflow_cols = workflow_df.columns

    life_cycle_col = get_first_existing_col(workflow_cols, ["life_cycle_state"])
    result_state_col = get_first_existing_col(workflow_cols, ["result_state"])
    failed_tasks_col = get_first_existing_col(workflow_cols, ["failed_tasks"])
    run_id_col = get_first_existing_col(workflow_cols, ["run_id"])
    job_id_col = get_first_existing_col(workflow_cols, ["job_id"])
    run_url_col = get_first_existing_col(workflow_cols, ["run_page_url"])
    checked_at_col = get_first_existing_col(workflow_cols, ["status_checked_at", "gold_created_at", "created_at"])

    if life_cycle_col and run_id_col:
        base = workflow_df

        if result_state_col is None:
            base = base.withColumn("result_state_tmp", F.lit(None).cast("string"))
            result_state_col = "result_state_tmp"

        if failed_tasks_col is None:
            base = base.withColumn("failed_tasks_tmp", F.lit(0))
            failed_tasks_col = "failed_tasks_tmp"

        if job_id_col is None:
            base = base.withColumn("job_id_tmp", F.lit(None).cast("string"))
            job_id_col = "job_id_tmp"

        if run_url_col is None:
            base = base.withColumn("run_url_tmp", F.lit(None).cast("string"))
            run_url_col = "run_url_tmp"

        if checked_at_col is None:
            base = base.withColumn("checked_at_tmp", F.current_timestamp())
            checked_at_col = "checked_at_tmp"

        latest_workflow = (
            base
            .orderBy(F.col(checked_at_col).desc())
            .limit(1)
        )

        workflow_alerts_df = (
            latest_workflow
            .filter(
                (F.col(result_state_col).isin("FAILED", "TIMEDOUT", "CANCELED"))
                | (F.col(failed_tasks_col).cast("int") > 0)
            )
            .withColumn("alert_type", F.lit("workflow_health"))
            .withColumn("alert_source", F.lit(workflow_status_table))
            .withColumn("risk_category", F.lit("data_platform"))
            .withColumn("severity", F.lit("critical"))
            .withColumn("severity_rank", F.lit(4))
            .withColumn("supplier_id", F.lit(None).cast("string"))
            .withColumn("supplier_name", F.lit(None).cast("string"))
            .withColumn("sku_id", F.lit(None).cast("string"))
            .withColumn("product_name", F.lit(None).cast("string"))
            .withColumn("risk_score", F.lit(None).cast("double"))
            .withColumn("risk_band", F.lit("critical"))
            .withColumn("trigger_metric_name", F.lit("failed_tasks"))
            .withColumn("trigger_metric_value", F.col(failed_tasks_col).cast("double"))
            .withColumn("trigger_threshold", F.lit(0.0))
            .withColumn("event_date", F.current_date().cast("string"))
            .withColumn("event_source", F.lit("lakeflow_job_status"))
            .withColumn("event_title", F.concat(F.lit("Lakeflow run unhealthy: "), F.col(run_id_col).cast("string")))
            .withColumn("event_url", F.col(run_url_col).cast("string"))
            .withColumn("alert_title", F.lit("Lakeflow workflow failure detected"))
            .withColumn(
                "alert_message",
                F.concat(
                    F.lit("Lakeflow run "),
                    F.col(run_id_col).cast("string"),
                    F.lit(" is unhealthy. lifecycle="),
                    F.col(life_cycle_col).cast("string"),
                    F.lit(", result="),
                    F.col(result_state_col).cast("string"),
                    F.lit(", failed_tasks="),
                    F.col(failed_tasks_col).cast("string")
                )
            )
            .withColumn("recommended_action", F.lit("Open the Lakeflow run, inspect failed task logs, and rerun from failed task after remediation."))
            .withColumn(
                "dedupe_key",
                F.concat_ws("|", F.lit("workflow_health"), F.col(run_id_col).cast("string"))
            )
            .withColumn("alert_id", stable_alert_id_expr("dedupe_key"))
        )

        workflow_alerts_df = add_common_alert_columns(workflow_alerts_df)
        workflow_alerts_df = align_to_alert_schema(workflow_alerts_df)

print("Workflow health alerts generated:", workflow_alerts_df.count())

# ─────────────────────────────────────────────────────────────
# 5. Agent serving health alerts
# ─────────────────────────────────────────────────────────────

serving_alerts_df = empty_alert_df()

serving_health_table = UPSTREAM_TABLES["serving_health"]

if table_exists(serving_health_table):
    serving_df = spark.table(serving_health_table)
    serving_cols = serving_df.columns

    overall_healthy_col = get_first_existing_col(serving_cols, ["overall_healthy"])
    failed_checks_col = get_first_existing_col(serving_cols, ["failed_checks"])
    health_run_id_col = get_first_existing_col(serving_cols, ["health_run_id"])
    checked_at_col = get_first_existing_col(serving_cols, ["health_finished_at", "created_at", "gold_created_at"])

    if overall_healthy_col:
        base = serving_df

        if failed_checks_col is None:
            base = base.withColumn("failed_checks_tmp", F.lit(0))
            failed_checks_col = "failed_checks_tmp"

        if health_run_id_col is None:
            base = base.withColumn("health_run_id_tmp", F.sha2(F.current_timestamp().cast("string"), 256))
            health_run_id_col = "health_run_id_tmp"

        if checked_at_col is None:
            base = base.withColumn("checked_at_tmp", F.current_timestamp())
            checked_at_col = "checked_at_tmp"

        latest_serving = (
            base
            .orderBy(F.col(checked_at_col).desc())
            .limit(1)
        )

        serving_alerts_df = (
            latest_serving
            .filter(
                (F.col(overall_healthy_col) == F.lit(False))
                | (F.col(failed_checks_col).cast("int") > 0)
            )
            .withColumn("alert_type", F.lit("agent_serving_health"))
            .withColumn("alert_source", F.lit(serving_health_table))
            .withColumn("risk_category", F.lit("ai_platform"))
            .withColumn("severity", F.lit("critical"))
            .withColumn("severity_rank", F.lit(4))
            .withColumn("supplier_id", F.lit(None).cast("string"))
            .withColumn("supplier_name", F.lit(None).cast("string"))
            .withColumn("sku_id", F.lit(None).cast("string"))
            .withColumn("product_name", F.lit(None).cast("string"))
            .withColumn("risk_score", F.lit(None).cast("double"))
            .withColumn("risk_band", F.lit("critical"))
            .withColumn("trigger_metric_name", F.lit("serving_failed_checks"))
            .withColumn("trigger_metric_value", F.col(failed_checks_col).cast("double"))
            .withColumn("trigger_threshold", F.lit(0.0))
            .withColumn("event_date", F.current_date().cast("string"))
            .withColumn("event_source", F.lit("agent_serving_health"))
            .withColumn("event_title", F.concat(F.lit("Agent serving unhealthy: "), F.col(health_run_id_col).cast("string")))
            .withColumn("event_url", F.lit(None).cast("string"))
            .withColumn("alert_title", F.lit("Agent serving health failure detected"))
            .withColumn(
                "alert_message",
                F.concat(
                    F.lit("Agent serving health check failed. health_run_id="),
                    F.col(health_run_id_col).cast("string"),
                    F.lit(", failed_checks="),
                    F.col(failed_checks_col).cast("string")
                )
            )
            .withColumn("recommended_action", F.lit("Inspect serving health output, validate runtime dependencies, and rerun the health check."))
            .withColumn(
                "dedupe_key",
                F.concat_ws("|", F.lit("agent_serving_health"), F.col(health_run_id_col).cast("string"))
            )
            .withColumn("alert_id", stable_alert_id_expr("dedupe_key"))
        )

        serving_alerts_df = add_common_alert_columns(serving_alerts_df)
        serving_alerts_df = align_to_alert_schema(serving_alerts_df)

print("Agent serving health alerts generated:", serving_alerts_df.count())

# ─────────────────────────────────────────────────────────────
# 6. Union, deduplicate, and save alerts
# ─────────────────────────────────────────────────────────────

all_alerts_df = (
    supplier_alerts_df
    .unionByName(sku_alerts_df)
    .unionByName(external_alerts_df)
    .unionByName(workflow_alerts_df)
    .unionByName(serving_alerts_df)
)

generated_alert_count = all_alerts_df.count()

print("\nTotal generated alerts before dedupe:", generated_alert_count)

if generated_alert_count > 0:
    # Drop duplicates within this run.
    all_alerts_deduped_df = all_alerts_df.dropDuplicates(["dedupe_key"])

    generated_deduped_count = all_alerts_deduped_df.count()

    print("Total generated alerts after in-run dedupe:", generated_deduped_count)

    all_alerts_deduped_df.createOrReplaceTempView("generated_alert_events")

    # Merge into alert events table by dedupe_key.
    # If the alert already exists, update status and source payload.
    spark.sql(f"""
    MERGE INTO {ALERT_EVENTS_TABLE} AS target
    USING generated_alert_events AS source
    ON target.dedupe_key = source.dedupe_key
    WHEN MATCHED THEN UPDATE SET
      target.alert_run_id = source.alert_run_id,
      target.alert_status = source.alert_status,
      target.is_active = source.is_active,
      target.created_at = source.created_at,
      target.source_payload_json = source.source_payload_json
    WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"Saved generated alerts to: {ALERT_EVENTS_TABLE}")

else:
    generated_deduped_count = 0
    print("No alert events generated in this run.")

# ─────────────────────────────────────────────────────────────
# 7. Alert generation summary
# ─────────────────────────────────────────────────────────────
# -------------------------------------------------------------------
# Schema-safe alert summary
# -------------------------------------------------------------------
# Some existing gold_alert_events versions do not include alert_run_id,
# alert_type, or risk_category. Do not fail Lakeflow on summary-only logic.
# This summary dynamically uses only columns that exist.

alert_events_df = spark.table(ALERT_EVENTS_TABLE)
alert_event_columns = set(alert_events_df.columns)

print("\nAlert events table columns:")
print(sorted(alert_event_columns))

# Filter to this run only if the table actually has alert_run_id.
# Current gold_alert_events does not, so we use all currently written alert events.
if "alert_run_id" in alert_event_columns:
    alert_events_for_summary_df = alert_events_df.filter(F.col("alert_run_id") == ALERT_RUN_ID)
    print(f"\nFiltering alert events for alert_run_id = {ALERT_RUN_ID}")
else:
    alert_events_for_summary_df = alert_events_df
    print("\nNo alert_run_id column found in gold_alert_events. Using full alert events table for summary.")

# Use only grouping columns that exist in the current table schema.
candidate_group_cols = [
    "alert_type",
    "severity",
    "risk_category",
    "entity_type",
    "status",
]

group_cols = [col_name for col_name in candidate_group_cols if col_name in alert_event_columns]

if group_cols:
    alert_type_counts = (
        alert_events_for_summary_df
        .groupBy(*group_cols)
        .agg(F.count("*").alias("alert_count"))
        .orderBy(F.desc("alert_count"))
    )
else:
    alert_type_counts = (
        alert_events_for_summary_df
        .agg(F.count("*").alias("alert_count"))
    )

print("\nAlert counts summary:")
display(alert_type_counts)

print("\nAlert counts for this run:")
display(alert_type_counts)

summary_payload = {
    "alert_run_id": ALERT_RUN_ID,
    "supplier_alerts": supplier_alerts_df.count(),
    "sku_stockout_alerts": sku_alerts_df.count(),
    "external_event_alerts": external_alerts_df.count(),
    "workflow_health_alerts": workflow_alerts_df.count(),
    "agent_serving_health_alerts": serving_alerts_df.count(),
    "generated_alert_count": generated_alert_count,
    "generated_deduped_count": generated_deduped_count,
    "alert_generation_started_at": alert_generation_started_at.isoformat(),
    "alert_generation_finished_at": datetime.now(timezone.utc).isoformat()
}

print("\nAlert generation summary:")
print(json.dumps(summary_payload, indent=2))

print("\nCell 2 complete.")
print("Next: Cell 3 will route generated alerts to recipients and create alert inbox items.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 32 — Cell 1A
# Repair gold_alert_events schema safely
# ─────────────────────────────────────────────────────────────

from datetime import datetime, timezone

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    DoubleType,
    TimestampType
)

assert "ALERT_EVENTS_TABLE" in globals(), "ALERT_EVENTS_TABLE missing. Rerun Cell 1."
assert "GOLD_SCHEMA" in globals(), "GOLD_SCHEMA missing. Rerun Cell 1."

LEGACY_ALERT_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_alert_events_legacy_pre_notebook32"

required_alert_event_columns = {
    "alert_id",
    "alert_run_id",
    "alert_type",
    "alert_source",
    "risk_category",
    "severity",
    "severity_rank",
    "supplier_id",
    "supplier_name",
    "sku_id",
    "product_name",
    "risk_score",
    "risk_band",
    "trigger_metric_name",
    "trigger_metric_value",
    "trigger_threshold",
    "event_date",
    "event_source",
    "event_title",
    "event_url",
    "alert_title",
    "alert_message",
    "recommended_action",
    "dedupe_key",
    "alert_status",
    "is_active",
    "created_at",
    "created_by",
    "source_payload_json"
}

alert_events_schema = StructType([
    StructField("alert_id", StringType(), False),
    StructField("alert_run_id", StringType(), True),
    StructField("alert_type", StringType(), True),
    StructField("alert_source", StringType(), True),
    StructField("risk_category", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("severity_rank", IntegerType(), True),

    StructField("supplier_id", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("sku_id", StringType(), True),
    StructField("product_name", StringType(), True),

    StructField("risk_score", DoubleType(), True),
    StructField("risk_band", StringType(), True),
    StructField("trigger_metric_name", StringType(), True),
    StructField("trigger_metric_value", DoubleType(), True),
    StructField("trigger_threshold", DoubleType(), True),

    StructField("event_date", StringType(), True),
    StructField("event_source", StringType(), True),
    StructField("event_title", StringType(), True),
    StructField("event_url", StringType(), True),

    StructField("alert_title", StringType(), True),
    StructField("alert_message", StringType(), True),
    StructField("recommended_action", StringType(), True),

    StructField("dedupe_key", StringType(), True),
    StructField("alert_status", StringType(), True),
    StructField("is_active", BooleanType(), True),

    StructField("created_at", TimestampType(), True),
    StructField("created_by", StringType(), True),
    StructField("source_payload_json", StringType(), True),
])

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False

if table_exists(ALERT_EVENTS_TABLE):
    existing_cols = set(spark.table(ALERT_EVENTS_TABLE).columns)
    missing_cols = sorted(required_alert_event_columns - existing_cols)

    print("Existing alert events columns:")
    print(sorted(existing_cols))

    print("\nMissing required columns:")
    print(missing_cols)

    if len(missing_cols) > 0:
        print("\nExisting gold_alert_events has legacy/incompatible schema.")

        if not table_exists(LEGACY_ALERT_EVENTS_TABLE):
            spark.sql(f"""
            CREATE TABLE {LEGACY_ALERT_EVENTS_TABLE}
            AS SELECT * FROM {ALERT_EVENTS_TABLE}
            """)
            print(f"Backed up legacy table to: {LEGACY_ALERT_EVENTS_TABLE}")
        else:
            print(f"Legacy backup already exists: {LEGACY_ALERT_EVENTS_TABLE}")

        spark.sql(f"DROP TABLE {ALERT_EVENTS_TABLE}")
        print(f"Dropped incompatible table: {ALERT_EVENTS_TABLE}")

        empty_alert_events_df = spark.createDataFrame([], schema=alert_events_schema)

        (
            empty_alert_events_df
            .write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .saveAsTable(ALERT_EVENTS_TABLE)
        )

        print(f"Recreated product alert events table: {ALERT_EVENTS_TABLE}")

    else:
        print("\nAlert events table already has the expected product schema.")

else:
    empty_alert_events_df = spark.createDataFrame([], schema=alert_events_schema)

    (
        empty_alert_events_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(ALERT_EVENTS_TABLE)
    )

    print(f"Created product alert events table: {ALERT_EVENTS_TABLE}")

print("\nFinal alert events schema:")
spark.table(ALERT_EVENTS_TABLE).printSchema()

print("\nCell 1A complete. Now rerun Cell 2.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 32 — Cell 2A
# Alert diagnostics + controlled smoke-test alerts when no real alerts exist
# ─────────────────────────────────────────────────────────────

import json
import uuid
import hashlib
from datetime import datetime, timezone

from pyspark.sql import functions as F

assert "ALERT_RUN_ID" in globals(), "ALERT_RUN_ID missing. Rerun Cell 1."
assert "ALERT_EVENTS_TABLE" in globals(), "ALERT_EVENTS_TABLE missing. Rerun Cell 1."
assert "UPSTREAM_TABLES" in globals(), "UPSTREAM_TABLES missing. Rerun Cell 1."

CREATE_SMOKE_TEST_ALERTS_WHEN_EMPTY = True

diagnostics_started_at = datetime.now(timezone.utc)

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def get_first_existing_col(columns, candidates):
    for col_name in candidates:
        if col_name in columns:
            return col_name
    return None


def make_alert_id(dedupe_key: str) -> str:
    return "alert_" + hashlib.sha256(dedupe_key.encode("utf-8")).hexdigest()[:24]


def safe_count(table_name: str):
    if not table_exists(table_name):
        return None
    return spark.table(table_name).count()


# ─────────────────────────────────────────────────────────────
# 1. Diagnostics: explain why zero alerts may be valid
# ─────────────────────────────────────────────────────────────

diagnostics = {
    "alert_run_id": ALERT_RUN_ID,
    "diagnostics_started_at": diagnostics_started_at.isoformat(),
    "tables": {}
}

# Supplier risk diagnostics
supplier_table = UPSTREAM_TABLES.get("supplier_risk_scores")

if supplier_table and table_exists(supplier_table):
    df = spark.table(supplier_table)
    cols = df.columns

    risk_score_col = get_first_existing_col(cols, ["supplier_risk_score", "risk_score", "final_supplier_risk_score"])
    risk_band_col = get_first_existing_col(cols, ["risk_band", "supplier_risk_band"])

    supplier_diag = {
        "table": supplier_table,
        "row_count": df.count(),
        "risk_score_col": risk_score_col,
        "risk_band_col": risk_band_col
    }

    if risk_score_col:
        row = (
            df
            .agg(
                F.max(F.col(risk_score_col)).alias("max_risk_score"),
                F.avg(F.col(risk_score_col)).alias("avg_risk_score"),
                F.sum(F.when(F.col(risk_score_col) >= 70.0, 1).otherwise(0)).alias("count_score_gte_70"),
                F.sum(F.when(F.col(risk_score_col) >= 45.0, 1).otherwise(0)).alias("count_score_gte_45")
            )
            .first()
        )

        supplier_diag.update({
            "max_risk_score": float(row["max_risk_score"]) if row["max_risk_score"] is not None else None,
            "avg_risk_score": float(row["avg_risk_score"]) if row["avg_risk_score"] is not None else None,
            "count_score_gte_70": int(row["count_score_gte_70"]) if row["count_score_gte_70"] is not None else 0,
            "count_score_gte_45": int(row["count_score_gte_45"]) if row["count_score_gte_45"] is not None else 0
        })

    diagnostics["tables"]["supplier_risk_scores"] = supplier_diag
else:
    diagnostics["tables"]["supplier_risk_scores"] = {
        "table": supplier_table,
        "exists": False
    }

# SKU stockout diagnostics
sku_table = UPSTREAM_TABLES.get("sku_stockout_risk_scores")

if sku_table and table_exists(sku_table):
    df = spark.table(sku_table)
    cols = df.columns

    stockout_score_col = get_first_existing_col(cols, ["sku_stockout_risk_score", "stockout_risk_score", "risk_score"])
    risk_band_col = get_first_existing_col(cols, ["stockout_risk_band", "risk_band"])

    sku_diag = {
        "table": sku_table,
        "row_count": df.count(),
        "stockout_score_col": stockout_score_col,
        "risk_band_col": risk_band_col
    }

    if stockout_score_col:
        row = (
            df
            .agg(
                F.max(F.col(stockout_score_col)).alias("max_stockout_score"),
                F.avg(F.col(stockout_score_col)).alias("avg_stockout_score"),
                F.sum(F.when(F.col(stockout_score_col) >= 0.75, 1).otherwise(0)).alias("count_score_gte_075"),
                F.sum(F.when(F.col(stockout_score_col) >= 0.50, 1).otherwise(0)).alias("count_score_gte_050")
            )
            .first()
        )

        sku_diag.update({
            "max_stockout_score": float(row["max_stockout_score"]) if row["max_stockout_score"] is not None else None,
            "avg_stockout_score": float(row["avg_stockout_score"]) if row["avg_stockout_score"] is not None else None,
            "count_score_gte_075": int(row["count_score_gte_075"]) if row["count_score_gte_075"] is not None else 0,
            "count_score_gte_050": int(row["count_score_gte_050"]) if row["count_score_gte_050"] is not None else 0
        })

    diagnostics["tables"]["sku_stockout_risk_scores"] = sku_diag
else:
    diagnostics["tables"]["sku_stockout_risk_scores"] = {
        "table": sku_table,
        "exists": False
    }

# External event diagnostics
external_table = UPSTREAM_TABLES.get("external_risk_event_mart")

if external_table and table_exists(external_table):
    df = spark.table(external_table)
    cols = df.columns

    severity_col = get_first_existing_col(cols, ["severity", "event_severity"])
    risk_category_col = get_first_existing_col(cols, ["risk_category", "event_category", "category"])

    external_diag = {
        "table": external_table,
        "row_count": df.count(),
        "severity_col": severity_col,
        "risk_category_col": risk_category_col
    }

    diagnostics["tables"]["external_risk_event_mart"] = external_diag

    if severity_col:
        print("\nExternal event severity distribution:")
        display(
            df
            .groupBy(F.lower(F.col(severity_col)).alias("severity"))
            .agg(F.count("*").alias("count"))
            .orderBy(F.desc("count"))
        )

    if risk_category_col:
        print("\nExternal event risk category distribution:")
        display(
            df
            .groupBy(F.col(risk_category_col).alias("risk_category"))
            .agg(F.count("*").alias("count"))
            .orderBy(F.desc("count"))
        )
else:
    diagnostics["tables"]["external_risk_event_mart"] = {
        "table": external_table,
        "exists": False
    }

# Workflow run diagnostics
workflow_table = UPSTREAM_TABLES.get("workflow_run_status_snapshots")

if workflow_table and table_exists(workflow_table):
    df = spark.table(workflow_table)

    latest_workflow = (
        df
        .orderBy(F.col("status_checked_at").desc() if "status_checked_at" in df.columns else F.col("gold_created_at").desc())
        .limit(1)
    )

    print("\nLatest workflow status snapshot:")
    display(latest_workflow)

    diagnostics["tables"]["workflow_run_status_snapshots"] = {
        "table": workflow_table,
        "row_count": df.count()
    }
else:
    diagnostics["tables"]["workflow_run_status_snapshots"] = {
        "table": workflow_table,
        "exists": False
    }

# Serving health diagnostics
serving_table = UPSTREAM_TABLES.get("serving_health")

if serving_table and table_exists(serving_table):
    df = spark.table(serving_table)

    order_col = "health_finished_at" if "health_finished_at" in df.columns else (
        "gold_created_at" if "gold_created_at" in df.columns else df.columns[0]
    )

    latest_serving = (
        df
        .orderBy(F.col(order_col).desc())
        .limit(1)
    )

    print("\nLatest serving health snapshot:")
    display(latest_serving)

    diagnostics["tables"]["serving_health"] = {
        "table": serving_table,
        "row_count": df.count()
    }
else:
    diagnostics["tables"]["serving_health"] = {
        "table": serving_table,
        "exists": False
    }

print("\nAlert diagnostics:")
print(json.dumps(diagnostics, indent=2, default=str))

# ─────────────────────────────────────────────────────────────
# 2. Check current generated alerts
# ─────────────────────────────────────────────────────────────

current_run_alert_count = (
    spark.table(ALERT_EVENTS_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
    .count()
)

print("\nCurrent alert count for this run:", current_run_alert_count)

# ─────────────────────────────────────────────────────────────
# 3. Create smoke-test alerts only if this run has no real alerts
# ─────────────────────────────────────────────────────────────

if current_run_alert_count == 0 and CREATE_SMOKE_TEST_ALERTS_WHEN_EMPTY:
    print("\nNo real alerts generated. Creating clearly labeled smoke-test alerts for routing validation.")

    created_at = datetime.now(timezone.utc)

    smoke_test_payload = {
        "is_smoke_test": True,
        "purpose": "Validate alert routing, inbox creation, and delivery logging when production thresholds have no active incidents.",
        "alert_run_id": ALERT_RUN_ID,
        "created_from_notebook": NOTEBOOK_NAME
    }

    smoke_rows = []

    # Smoke test 1: cyber external event route -> security_team / Siddharath
    dedupe_key_1 = f"smoke_test|external_event|cyber|{ALERT_RUN_ID}"

    smoke_rows.append({
        "alert_id": make_alert_id(dedupe_key_1),
        "alert_run_id": ALERT_RUN_ID,
        "alert_type": "external_event",
        "alert_source": "notebook32_smoke_test",
        "risk_category": "cyber",
        "severity": "high",
        "severity_rank": 3,

        "supplier_id": "SUP_134",
        "supplier_name": "FlexPack Industries",
        "sku_id": None,
        "product_name": None,

        "risk_score": None,
        "risk_band": "high",
        "trigger_metric_name": "smoke_test_external_event_severity_rank",
        "trigger_metric_value": 3.0,
        "trigger_threshold": 3.0,

        "event_date": created_at.date().isoformat(),
        "event_source": "smoke_test_cisa_route",
        "event_title": "Smoke test: high cyber supplier event",
        "event_url": None,

        "alert_title": "SMOKE TEST: High cyber event route validation",
        "alert_message": "This is a controlled smoke-test alert to validate cyber alert routing to the security team. It is not a real supplier incident.",
        "recommended_action": "Validate that this alert routes to the configured security-team recipient and appears in the alert inbox.",

        "dedupe_key": dedupe_key_1,
        "alert_status": "new",
        "is_active": True,

        "created_at": created_at,
        "created_by": CREATED_BY,
        "source_payload_json": json.dumps(smoke_test_payload, indent=2)
    })

    # Smoke test 2: supplier risk route -> supplier_ops_team / Vigya
    dedupe_key_2 = f"smoke_test|supplier_risk|supplier_ops|{ALERT_RUN_ID}"

    smoke_rows.append({
        "alert_id": make_alert_id(dedupe_key_2),
        "alert_run_id": ALERT_RUN_ID,
        "alert_type": "supplier_risk",
        "alert_source": "notebook32_smoke_test",
        "risk_category": "supplier_risk",
        "severity": "high",
        "severity_rank": 3,

        "supplier_id": "SUP_134",
        "supplier_name": "FlexPack Industries",
        "sku_id": None,
        "product_name": None,

        "risk_score": 76.0,
        "risk_band": "high",
        "trigger_metric_name": "smoke_test_supplier_risk_score",
        "trigger_metric_value": 76.0,
        "trigger_threshold": 70.0,

        "event_date": created_at.date().isoformat(),
        "event_source": "smoke_test_supplier_risk_route",
        "event_title": "Smoke test: high supplier risk threshold",
        "event_url": None,

        "alert_title": "SMOKE TEST: High supplier risk route validation",
        "alert_message": "This is a controlled smoke-test alert to validate supplier-risk routing to the supplier operations team. It is not a real supplier incident.",
        "recommended_action": "Validate that this alert routes to the configured supplier-ops recipient and appears in the alert inbox.",

        "dedupe_key": dedupe_key_2,
        "alert_status": "new",
        "is_active": True,

        "created_at": created_at,
        "created_by": CREATED_BY,
        "source_payload_json": json.dumps(smoke_test_payload, indent=2)
    })

    target_schema = spark.table(ALERT_EVENTS_TABLE).schema

    smoke_df = spark.createDataFrame(smoke_rows, schema=target_schema)
    smoke_df.createOrReplaceTempView("smoke_test_alert_events")

    spark.sql(f"""
    MERGE INTO {ALERT_EVENTS_TABLE} AS target
    USING smoke_test_alert_events AS source
    ON target.dedupe_key = source.dedupe_key
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"Inserted smoke-test alerts into: {ALERT_EVENTS_TABLE}")

else:
    print("\nSmoke-test alert creation skipped.")

# ─────────────────────────────────────────────────────────────
# 4. Preview alert events for this run
# ─────────────────────────────────────────────────────────────

final_current_run_alerts = (
    spark.table(ALERT_EVENTS_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
)

print("\nFinal alert count for this run:", final_current_run_alerts.count())

display(
    final_current_run_alerts
    .select(
        "alert_id",
        "alert_type",
        "risk_category",
        "severity",
        "supplier_id",
        "supplier_name",
        "alert_title",
        "alert_status",
        "alert_source",
        "created_at"
    )
    .orderBy(F.desc("created_at"))
)

print("\nCell 2A complete.")
print("Next: Cell 3 will route alert events to recipients and create alert inbox + delivery log records.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 32 — Cell 3
# Route alert events to recipients and create inbox + delivery log records
# ─────────────────────────────────────────────────────────────

import json
import hashlib
from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F

assert "ALERT_RUN_ID" in globals(), "ALERT_RUN_ID missing. Rerun Cell 1."
assert "ALERT_EVENTS_TABLE" in globals(), "ALERT_EVENTS_TABLE missing. Rerun Cell 1."
assert "ALERT_ROUTING_CONFIG_TABLE" in globals(), "ALERT_ROUTING_CONFIG_TABLE missing. Rerun Cell 1."
assert "ALERT_INBOX_TABLE" in globals(), "ALERT_INBOX_TABLE missing. Rerun Cell 1."
assert "ALERT_DELIVERY_LOG_TABLE" in globals(), "ALERT_DELIVERY_LOG_TABLE missing. Rerun Cell 1."

routing_started_at = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def align_to_table_schema(df, table_name: str):
    """
    Aligns DataFrame columns to an existing Delta table schema.
    Adds missing columns as null and casts existing columns.
    """
    target_schema = spark.table(table_name).schema
    target_cols = [field.name for field in target_schema]

    aligned_df = df

    for field in target_schema:
        if field.name not in aligned_df.columns:
            aligned_df = aligned_df.withColumn(field.name, F.lit(None).cast(field.dataType))
        else:
            aligned_df = aligned_df.withColumn(field.name, F.col(field.name).cast(field.dataType))

    return aligned_df.select(target_cols)


# ─────────────────────────────────────────────────────────────
# 1. Load new alert events for this run
# ─────────────────────────────────────────────────────────────

alerts_df = (
    spark.table(ALERT_EVENTS_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
    .filter(F.col("is_active") == True)
    .filter(F.col("alert_status").isin("new", "open", "routed"))
)

alert_count = alerts_df.count()

print("Alert events available for routing:", alert_count)

if alert_count == 0:
    print("No alerts found for this ALERT_RUN_ID. Nothing to route.")
    dbutils.notebook.exit("No alerts to route.")

display(
    alerts_df
    .select(
        "alert_id",
        "alert_type",
        "risk_category",
        "severity",
        "severity_rank",
        "supplier_id",
        "sku_id",
        "alert_title",
        "alert_source"
    )
    .orderBy(F.desc("severity_rank"), "alert_type")
)

# ─────────────────────────────────────────────────────────────
# 2. Load active routing rules
# ─────────────────────────────────────────────────────────────

routes_df = (
    spark.table(ALERT_ROUTING_CONFIG_TABLE)
    .filter(F.col("is_active") == True)
)

route_count = routes_df.count()

print("Active routing rules:", route_count)

if route_count == 0:
    raise ValueError("No active routing rules found.")

# ─────────────────────────────────────────────────────────────
# 3. Match alerts to routing rules
# ─────────────────────────────────────────────────────────────
# Matching rules:
# - alert_type can match exact value or wildcard '*'
# - risk_category can match exact value or wildcard '*'
# - supplier_id can match exact value, wildcard '*', or null
# - sku_id can match exact value, wildcard '*', or null
# - alert severity_rank must be >= routing minimum_severity_rank
# - use lowest rule_priority as the best route per alert/channel/recipient

a = alerts_df.alias("a")
r = routes_df.alias("r")

matched_routes_df = (
    a.join(
        r,
        (
            ((F.col("r.alert_type") == F.col("a.alert_type")) | (F.col("r.alert_type") == F.lit("*")))
            & ((F.col("r.risk_category") == F.col("a.risk_category")) | (F.col("r.risk_category") == F.lit("*")))
            & (
                (F.col("r.supplier_id") == F.col("a.supplier_id"))
                | (F.col("r.supplier_id") == F.lit("*"))
                | F.col("r.supplier_id").isNull()
            )
            & (
                (F.col("r.sku_id") == F.col("a.sku_id"))
                | (F.col("r.sku_id") == F.lit("*"))
                | F.col("r.sku_id").isNull()
            )
            & (F.col("a.severity_rank") >= F.coalesce(F.col("r.minimum_severity_rank"), F.lit(1)))
        ),
        "inner"
    )
    .select(
        F.col("a.alert_id"),
        F.col("a.alert_run_id"),
        F.col("a.alert_type"),
        F.col("a.alert_source"),
        F.col("a.risk_category"),
        F.col("a.severity"),
        F.col("a.severity_rank"),
        F.col("a.supplier_id"),
        F.col("a.supplier_name"),
        F.col("a.sku_id"),
        F.col("a.product_name"),
        F.col("a.risk_score"),
        F.col("a.risk_band"),
        F.col("a.alert_title"),
        F.col("a.alert_message"),
        F.col("a.recommended_action"),
        F.col("a.dedupe_key"),
        F.col("a.created_at").alias("alert_created_at"),
        F.col("r.routing_rule_id"),
        F.col("r.rule_priority"),
        F.col("r.recipient_group"),
        F.col("r.recipient_name"),
        F.col("r.recipient_email"),
        F.col("r.channel"),
        F.col("r.send_enabled"),
        F.col("r.escalation_minutes")
    )
)

matched_count = matched_routes_df.count()

print("Matched alert-route rows before best-rule selection:", matched_count)

if matched_count == 0:
    raise ValueError(
        "Alerts exist, but no routing rules matched. "
        "Check alert_type, risk_category, severity_rank, and routing config."
    )

# Pick the best matching route per alert + recipient + channel.
# This prevents default fallback rules from duplicating more specific routes.
best_route_window = (
    Window
    .partitionBy("alert_id", "recipient_email", "channel")
    .orderBy(F.col("rule_priority").asc())
)

routed_alerts_df = (
    matched_routes_df
    .withColumn("route_rank", F.row_number().over(best_route_window))
    .filter(F.col("route_rank") == 1)
    .drop("route_rank")
)

routed_count = routed_alerts_df.count()

print("Routed alert-recipient rows after best-rule selection:", routed_count)

display(
    routed_alerts_df
    .select(
        "alert_id",
        "alert_type",
        "risk_category",
        "severity",
        "routing_rule_id",
        "recipient_group",
        "recipient_name",
        "recipient_email",
        "channel",
        "send_enabled"
    )
    .orderBy("alert_type", "recipient_group")
)

# ─────────────────────────────────────────────────────────────
# 4. Create alert inbox records
# ─────────────────────────────────────────────────────────────

inbox_df = (
    routed_alerts_df
    .withColumn(
        "inbox_item_id",
        F.concat(
            F.lit("inbox_"),
            F.sha2(
                F.concat_ws(
                    "|",
                    F.col("alert_id"),
                    F.col("recipient_email"),
                    F.col("channel"),
                    F.lit(ALERT_RUN_ID)
                ),
                256
            ).substr(1, 24)
        )
    )
    .withColumn("inbox_status", F.lit("new"))
    .withColumn("assigned_to", F.col("recipient_email"))
    .withColumn("acknowledged_at", F.lit(None).cast("timestamp"))
    .withColumn("resolved_at", F.lit(None).cast("timestamp"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("created_by", F.lit(CREATED_BY))
    .select(
        "inbox_item_id",
        "alert_id",
        "alert_run_id",
        "alert_type",
        "risk_category",
        "severity",
        "severity_rank",
        "supplier_id",
        "supplier_name",
        "sku_id",
        "recipient_group",
        "recipient_name",
        "recipient_email",
        "channel",
        "alert_title",
        "alert_message",
        "recommended_action",
        "inbox_status",
        "assigned_to",
        "acknowledged_at",
        "resolved_at",
        "created_at",
        "created_by"
    )
)

inbox_df = align_to_table_schema(inbox_df, ALERT_INBOX_TABLE)

inbox_df.createOrReplaceTempView("generated_alert_inbox_items")

spark.sql(f"""
MERGE INTO {ALERT_INBOX_TABLE} AS target
USING generated_alert_inbox_items AS source
ON target.inbox_item_id = source.inbox_item_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print(f"Saved alert inbox items to: {ALERT_INBOX_TABLE}")

# ─────────────────────────────────────────────────────────────
# 5. Create delivery log records
# ─────────────────────────────────────────────────────────────
# Email sending is intentionally disabled unless send_enabled=True.
# For now, this logs what would have been sent.

delivery_df = (
    routed_alerts_df
    .withColumn(
        "inbox_item_id",
        F.concat(
            F.lit("inbox_"),
            F.sha2(
                F.concat_ws(
                    "|",
                    F.col("alert_id"),
                    F.col("recipient_email"),
                    F.col("channel"),
                    F.lit(ALERT_RUN_ID)
                ),
                256
            ).substr(1, 24)
        )
    )
    .withColumn(
        "delivery_id",
        F.concat(
            F.lit("delivery_"),
            F.sha2(
                F.concat_ws(
                    "|",
                    F.col("alert_id"),
                    F.col("recipient_email"),
                    F.col("channel"),
                    F.lit(ALERT_RUN_ID)
                ),
                256
            ).substr(1, 24)
        )
    )
    .withColumn(
        "message_subject",
        F.concat(
            F.lit("[SupplySage AI] "),
            F.upper(F.col("severity")),
            F.lit(" alert: "),
            F.col("alert_title")
        )
    )
    .withColumn(
        "message_body",
        F.concat(
            F.lit("SupplySage AI alert notification\n\n"),
            F.lit("Alert type: "),
            F.col("alert_type"),
            F.lit("\nRisk category: "),
            F.col("risk_category"),
            F.lit("\nSeverity: "),
            F.col("severity"),
            F.lit("\nSupplier: "),
            F.coalesce(F.col("supplier_name"), F.lit("N/A")),
            F.lit("\nSKU: "),
            F.coalesce(F.col("sku_id"), F.lit("N/A")),
            F.lit("\n\nMessage:\n"),
            F.col("alert_message"),
            F.lit("\n\nRecommended action:\n"),
            F.col("recommended_action"),
            F.lit("\n\nRouting rule: "),
            F.col("routing_rule_id"),
            F.lit("\n\nNote: delivery is currently logged only because send_enabled=false.")
        )
    )
    .withColumn("delivery_attempted", F.when(F.col("send_enabled") == True, F.lit(True)).otherwise(F.lit(False)))
    .withColumn(
        "delivery_status",
        F.when(F.col("send_enabled") == True, F.lit("ready_to_send"))
        .otherwise(F.lit("logged_not_sent_send_disabled"))
    )
    .withColumn("delivery_error", F.lit(None).cast("string"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("delivered_at", F.when(F.col("send_enabled") == True, F.current_timestamp()).otherwise(F.lit(None).cast("timestamp")))
    .withColumn("created_by", F.lit(CREATED_BY))
    .select(
        "delivery_id",
        "alert_id",
        "inbox_item_id",
        "alert_run_id",
        "recipient_group",
        "recipient_name",
        "recipient_email",
        "channel",
        "send_enabled",
        "delivery_status",
        "delivery_attempted",
        "delivery_error",
        "message_subject",
        "message_body",
        "created_at",
        "delivered_at",
        "created_by"
    )
)

delivery_df = align_to_table_schema(delivery_df, ALERT_DELIVERY_LOG_TABLE)

delivery_df.createOrReplaceTempView("generated_alert_delivery_log")

spark.sql(f"""
MERGE INTO {ALERT_DELIVERY_LOG_TABLE} AS target
USING generated_alert_delivery_log AS source
ON target.delivery_id = source.delivery_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print(f"Saved alert delivery log records to: {ALERT_DELIVERY_LOG_TABLE}")

# ─────────────────────────────────────────────────────────────
# 6. Update alert event status to routed
# ─────────────────────────────────────────────────────────────

routed_alert_ids_df = (
    routed_alerts_df
    .select("alert_id")
    .distinct()
    .withColumn("new_alert_status", F.lit("routed"))
)

routed_alert_ids_df.createOrReplaceTempView("routed_alert_ids")

spark.sql(f"""
MERGE INTO {ALERT_EVENTS_TABLE} AS target
USING routed_alert_ids AS source
ON target.alert_id = source.alert_id
WHEN MATCHED THEN UPDATE SET
  target.alert_status = source.new_alert_status
""")

print(f"Updated routed alert status in: {ALERT_EVENTS_TABLE}")

# ─────────────────────────────────────────────────────────────
# 7. Routing summary
# ─────────────────────────────────────────────────────────────

routing_finished_at = datetime.now(timezone.utc)

routing_summary = {
    "alert_run_id": ALERT_RUN_ID,
    "alerts_available_for_routing": alert_count,
    "active_routing_rules": route_count,
    "matched_alert_route_rows": matched_count,
    "routed_alert_recipient_rows": routed_count,
    "inbox_items_created_or_updated": inbox_df.count(),
    "delivery_logs_created_or_updated": delivery_df.count(),
    "emails_actually_sent": int(delivery_df.filter(F.col("delivery_attempted") == True).count()),
    "emails_logged_not_sent": int(delivery_df.filter(F.col("delivery_attempted") == False).count()),
    "routing_started_at": routing_started_at.isoformat(),
    "routing_finished_at": routing_finished_at.isoformat()
}

print("\nAlert routing summary:")
print(json.dumps(routing_summary, indent=2))

print("\nAlert inbox preview:")
display(
    spark.table(ALERT_INBOX_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
    .select(
        "inbox_item_id",
        "alert_type",
        "risk_category",
        "severity",
        "supplier_name",
        "recipient_group",
        "recipient_name",
        "recipient_email",
        "channel",
        "inbox_status",
        "alert_title"
    )
    .orderBy("recipient_group", "alert_type")
)

print("\nDelivery log preview:")
display(
    spark.table(ALERT_DELIVERY_LOG_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
    .select(
        "delivery_id",
        "alert_id",
        "recipient_group",
        "recipient_name",
        "recipient_email",
        "channel",
        "send_enabled",
        "delivery_status",
        "delivery_attempted",
        "message_subject"
    )
    .orderBy("recipient_group")
)

print("\nCell 3 complete.")
print("Next: Cell 4 will create an alerting summary table for the UI and portfolio proof.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 32 — Cell 4A
# Fix UI breakdown metric_value type and rebuild UI breakdown
# ─────────────────────────────────────────────────────────────

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)

assert "ALERT_RUN_ID" in globals(), "ALERT_RUN_ID missing. Rerun Cell 1."
assert "ALERT_EVENTS_TABLE" in globals(), "ALERT_EVENTS_TABLE missing. Rerun Cell 1."
assert "ALERT_INBOX_TABLE" in globals(), "ALERT_INBOX_TABLE missing. Rerun Cell 1."
assert "ALERT_DELIVERY_LOG_TABLE" in globals(), "ALERT_DELIVERY_LOG_TABLE missing. Rerun Cell 1."
assert "GOLD_SCHEMA" in globals(), "GOLD_SCHEMA missing. Rerun Cell 1."

ALERTING_UI_BREAKDOWN_TABLE = f"{GOLD_SCHEMA}.gold_alerting_ui_breakdown"
LEGACY_UI_BREAKDOWN_TABLE = f"{GOLD_SCHEMA}.gold_alerting_ui_breakdown_legacy_bad_metric_type"

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False

# ─────────────────────────────────────────────────────────────
# 1. Backup old table if needed, then recreate with LongType metric_value
# ─────────────────────────────────────────────────────────────

ui_breakdown_schema = StructType([
    StructField("alert_run_id", StringType(), False),
    StructField("breakdown_type", StringType(), True),
    StructField("breakdown_key", StringType(), True),
    StructField("breakdown_label", StringType(), True),
    StructField("alert_type", StringType(), True),
    StructField("risk_category", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("recipient_group", StringType(), True),
    StructField("recipient_email", StringType(), True),
    StructField("metric_name", StringType(), True),
    StructField("metric_value", LongType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("created_by", StringType(), True),
])

if table_exists(ALERTING_UI_BREAKDOWN_TABLE):
    existing_schema = spark.table(ALERTING_UI_BREAKDOWN_TABLE).schema
    metric_field = [field for field in existing_schema if field.name == "metric_value"]

    print("Existing metric_value field:")
    print(metric_field[0] if metric_field else "metric_value not found")

    if not table_exists(LEGACY_UI_BREAKDOWN_TABLE):
        spark.sql(f"""
        CREATE TABLE {LEGACY_UI_BREAKDOWN_TABLE}
        AS SELECT * FROM {ALERTING_UI_BREAKDOWN_TABLE}
        """)
        print(f"Backed up old UI breakdown table to: {LEGACY_UI_BREAKDOWN_TABLE}")
    else:
        print(f"Legacy backup already exists: {LEGACY_UI_BREAKDOWN_TABLE}")

    spark.sql(f"DROP TABLE {ALERTING_UI_BREAKDOWN_TABLE}")
    print(f"Dropped old UI breakdown table: {ALERTING_UI_BREAKDOWN_TABLE}")

empty_ui_df = spark.createDataFrame([], schema=ui_breakdown_schema)

(
    empty_ui_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(ALERTING_UI_BREAKDOWN_TABLE)
)

print(f"Recreated UI breakdown table with LongType metric_value: {ALERTING_UI_BREAKDOWN_TABLE}")

# ─────────────────────────────────────────────────────────────
# 2. Reload run-scoped data
# ─────────────────────────────────────────────────────────────

alerts_df = (
    spark.table(ALERT_EVENTS_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
)

inbox_df = (
    spark.table(ALERT_INBOX_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
)

delivery_df = (
    spark.table(ALERT_DELIVERY_LOG_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
)

print("Alerts:", alerts_df.count())
print("Inbox items:", inbox_df.count())
print("Delivery logs:", delivery_df.count())

# ─────────────────────────────────────────────────────────────
# 3. Rebuild UI breakdown with metric_value cast to LongType
# ─────────────────────────────────────────────────────────────

alert_type_breakdown_df = (
    alerts_df
    .groupBy("alert_type", "risk_category", "severity")
    .agg(F.count("*").cast("long").alias("metric_value"))
    .withColumn("alert_run_id", F.lit(ALERT_RUN_ID))
    .withColumn("breakdown_type", F.lit("alert_type_risk_category_severity"))
    .withColumn(
        "breakdown_key",
        F.concat_ws("|", F.col("alert_type"), F.col("risk_category"), F.col("severity"))
    )
    .withColumn(
        "breakdown_label",
        F.concat_ws(" / ", F.col("alert_type"), F.col("risk_category"), F.col("severity"))
    )
    .withColumn("recipient_group", F.lit(None).cast("string"))
    .withColumn("recipient_email", F.lit(None).cast("string"))
    .withColumn("metric_name", F.lit("alert_count"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("created_by", F.lit(CREATED_BY))
    .select(
        "alert_run_id",
        "breakdown_type",
        "breakdown_key",
        "breakdown_label",
        "alert_type",
        "risk_category",
        "severity",
        "recipient_group",
        "recipient_email",
        "metric_name",
        "metric_value",
        "created_at",
        "created_by"
    )
)

recipient_breakdown_df = (
    inbox_df
    .groupBy("recipient_group", "recipient_email")
    .agg(F.count("*").cast("long").alias("metric_value"))
    .withColumn("alert_run_id", F.lit(ALERT_RUN_ID))
    .withColumn("breakdown_type", F.lit("recipient_group_email"))
    .withColumn(
        "breakdown_key",
        F.concat_ws("|", F.col("recipient_group"), F.col("recipient_email"))
    )
    .withColumn(
        "breakdown_label",
        F.concat_ws(" / ", F.col("recipient_group"), F.col("recipient_email"))
    )
    .withColumn("alert_type", F.lit(None).cast("string"))
    .withColumn("risk_category", F.lit(None).cast("string"))
    .withColumn("severity", F.lit(None).cast("string"))
    .withColumn("metric_name", F.lit("inbox_item_count"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("created_by", F.lit(CREATED_BY))
    .select(
        "alert_run_id",
        "breakdown_type",
        "breakdown_key",
        "breakdown_label",
        "alert_type",
        "risk_category",
        "severity",
        "recipient_group",
        "recipient_email",
        "metric_name",
        "metric_value",
        "created_at",
        "created_by"
    )
)

delivery_breakdown_df = (
    delivery_df
    .groupBy("delivery_status")
    .agg(F.count("*").cast("long").alias("metric_value"))
    .withColumn("alert_run_id", F.lit(ALERT_RUN_ID))
    .withColumn("breakdown_type", F.lit("delivery_status"))
    .withColumn("breakdown_key", F.col("delivery_status"))
    .withColumn("breakdown_label", F.col("delivery_status"))
    .withColumn("alert_type", F.lit(None).cast("string"))
    .withColumn("risk_category", F.lit(None).cast("string"))
    .withColumn("severity", F.lit(None).cast("string"))
    .withColumn("recipient_group", F.lit(None).cast("string"))
    .withColumn("recipient_email", F.lit(None).cast("string"))
    .withColumn("metric_name", F.lit("delivery_log_count"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("created_by", F.lit(CREATED_BY))
    .select(
        "alert_run_id",
        "breakdown_type",
        "breakdown_key",
        "breakdown_label",
        "alert_type",
        "risk_category",
        "severity",
        "recipient_group",
        "recipient_email",
        "metric_name",
        "metric_value",
        "created_at",
        "created_by"
    )
)

ui_breakdown_df = (
    alert_type_breakdown_df
    .unionByName(recipient_breakdown_df)
    .unionByName(delivery_breakdown_df)
)

(
    ui_breakdown_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(ALERTING_UI_BREAKDOWN_TABLE)
)

print(f"Saved rebuilt UI breakdown to: {ALERTING_UI_BREAKDOWN_TABLE}")

# ─────────────────────────────────────────────────────────────
# 4. Preview final proof
# ─────────────────────────────────────────────────────────────

print("\nUI breakdown preview:")
display(
    spark.table(ALERTING_UI_BREAKDOWN_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
    .orderBy("breakdown_type", "breakdown_label")
)

print("\nAlert inbox proof:")
display(
    spark.table(ALERT_INBOX_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
    .select(
        "recipient_group",
        "recipient_name",
        "recipient_email",
        "alert_type",
        "risk_category",
        "severity",
        "inbox_status",
        "alert_title"
    )
    .orderBy("recipient_group")
)

print("\nDelivery log proof:")
display(
    spark.table(ALERT_DELIVERY_LOG_TABLE)
    .filter(F.col("alert_run_id") == ALERT_RUN_ID)
    .select(
        "recipient_group",
        "recipient_name",
        "recipient_email",
        "channel",
        "send_enabled",
        "delivery_status",
        "delivery_attempted",
        "message_subject"
    )
    .orderBy("recipient_group")
)

print("\nCell 4A complete.")
print("Notebook 32 alerting summary layer is now UI-ready.")